# Level 2 프로젝트 스타터: Vision-Guided Sorting Agent

# 프로젝트 스타터

과제:

> Find and sort the six cubes in the warehouse into their matching destination pads.

제출 에이전트 규칙:

- 카메라 관찰값, 고정 자연어 과제, `robot_status`, action result, 허용된 SDK skill, structured LLM decision을 사용하세요.
- `scene_state`, 정답 좌표, 정확한 cube/pad entity ID, 고정 action sequence는 사용할 수 없습니다.
- LLM-assisted observe-decide-act loop를 유지하고 observation/LLM decision/action/outcome을 기록하세요.

Level 2 규칙: `go_to`는 사용할 수 없습니다. `set_head`, 카메라 관찰값, `set_velocity`, memory, recovery behavior로 이동해야 합니다.

## Editing Guide

이 노트북은 완성된 해답이 아니라 스타터입니다.

- **Support code**: wrapper, schema validation, setup, baseline perception입니다. 먼저 읽어보고 대부분은 그대로 두어도 됩니다.
- **Student TODO**: 팀이 직접 설계하고 개선하고 테스트한 뒤 발표에서 설명해야 하는 부분입니다.
- 완전히 다른 구조를 사용해도 되지만, 프로젝트 규칙은 지켜야 합니다.

In [ ]:
# Colab/local setup. Run this once.
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv opencv-python Pillow matplotlib requests

In [ ]:
import asyncio
import math
import os
import time
from dataclasses import dataclass, field
from typing import Any

import cv2
import numpy as np
from IPython.display import Image, display

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key


def load_keys():
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
    except Exception:
        pass

    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY", userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY", userdata.get("TOKAMAK_API_KEY") or "")
    except Exception:
        pass


load_keys()

MENLO_API_KEY = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL = "https://platform-auth.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY, "Set MENLO_API_KEY in Colab Secrets or a local .env file."
print("MENLO_API_KEY loaded")
print(f"TOKAMAK_API_KEY: {'set' if TOKAMAK_API_KEY else 'not set'}")

## Required LLM Decision Schema

물리적 과제 문장은 고정되어 있지만, LLM은 실행 중 high-level decision loop에 반드시 참여해야 합니다. LLM은 observation, memory, recent outcome을 바탕으로 다음 행동을 결정해야 합니다.

LLM은 다음과 같은 JSON을 반환해야 합니다.

```json
{
  "next_action": "search_cube",
  "target_color": "red",
  "reason": "Red is visible and has not been completed."
}
```

허용되는 `next_action` 값:

```text
search_cube
navigate_to_cube
pick_cube
search_pad
navigate_to_pad
place_cube
recover
skip_target
stop
```

In [ ]:
import json
from dataclasses import dataclass, field
from typing import Any

TASK = "Find and sort the six cubes in the warehouse into their matching destination pads."

ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


@dataclass
class AgentDecision:
    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    delivered_count: int = 0
    held_color: str | None = None
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_colors: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    robot_status: Any
    detections: list[dict[str, Any]]
    note: str = ""


def parse_agent_decision(text):
    """Support code: validate the structured JSON returned by the LLM."""
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None

    if data.get("next_action") not in ALLOWED_NEXT_ACTIONS:
        return None
    if data.get("target_color") is not None and not isinstance(data.get("target_color"), str):
        return None

    return AgentDecision(
        next_action=data["next_action"],
        target_color=data.get("target_color"),
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(task, observation, memory, last_result=None):
    """Support code: compact robot state for the text LLM."""
    visible = [
        {
            "color": detection["color"],
            "angle_deg": detection["angle_deg"],
            "blob_area": detection["blob_area"],
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "delivered_count": memory.delivered_count,
        "completed_colors": memory.completed_colors,
        "failed_attempts": memory.failed_attempts,
        "last_result": last_result,
        "note": observation.note,
    }


In [ ]:
client = AsyncClient(base_url=RCS_URL, api_key=MENLO_API_KEY)
room_key = generate_room_key(prefix="hansung-project")
robot = await client.create_robot(room_key=room_key)
robot_id = robot.id

viewer_url = f"{VIEWER_BASE_URL}/?room_key={room_key}"
print("Open viewer in Chrome:")
print(viewer_url)

session = await connect(
    client,
    robot_id,
    worker_names=[],
    rcw_identity_prefix="simplesim",
    join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join or skills were not discovered. Is the Chrome tab open?")


skills = await wait_for_skills()
print("Skills found:", [skill.name for skill in skills])

## Support Code: Safe Project Helpers

이 wrapper들은 프로젝트에서 허용되는 SDK call만 노출합니다. 워크숍 setup code를 반복해서 작성하지 않도록 제공됩니다.

In [ ]:
# ---------------------------------------------------------------------------
# SUPPORT CODE: 프로젝트에서 허용되는 SDK wrapper
# ---------------------------------------------------------------------------
# These helpers avoid restricted state and exact cube/pad IDs. Most teams can
# leave this section as-is and build their strategy in the TODO cells below.

async def get_robot_status():
    """robot pose, motion status, neck state를 읽습니다."""
    return await session.state.get("robot_status")


async def get_camera_frame():
    """POV camera frame을 캡처합니다."""
    return await session.get_vision("pov")


async def forbidden_scene_state():
    raise RuntimeError("scene_state is not allowed in submitted project agents.")


async def show_camera():
    jpeg = await get_camera_frame()
    display(Image(data=jpeg))
    return jpeg


async def set_head(yaw=None, pitch=None):
    """walking direction은 바꾸지 않고 camera 방향만 조준합니다."""
    args = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    return await session.invoke("set_head", args, timeout_s=10)


async def move_velocity(vx=0.0, vy=0.0, wz=0.0, duration_s=1.0):
    """짧은 body-frame velocity command를 보내고 정지합니다."""
    return await session.invoke(
        "set_velocity",
        {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
        timeout_s=30,
    )


async def cancel_action():
    return await session.invoke("cancel", {})


async def pick_nearest_cube():
    """Pick after your code has visually positioned near the intended cube."""
    return await session.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": "cube"}},
        timeout_s=300,
    )


async def place_nearest_zone():
    """Place after your code has visually reached the matching pad."""
    return await session.invoke("place_entity", {}, timeout_s=300)


def result_summary(result):
    error = getattr(result, "error", None)
    return {
        "status": getattr(result, "status", None),
        "error": getattr(error, "message", None) if error else None,
    }


## Support Code: Perception Baseline

Workshop 2 스타일의 기본 detector입니다. 더 강한 perception이 필요하면 개선하거나 교체하세요.

In [ ]:
# ---------------------------------------------------------------------------
# SUPPORT CODE: Workshop 2 기반 기본 color-blob baseline
# ---------------------------------------------------------------------------
# You may improve or replace this detector. Keep submitted code observation-based.

HFOV_HALF_DEG = 30.0
MIN_BLOB_AREA = 200

COLOR_RANGES = {
    "red": [
        (np.array([0, 50, 50]), np.array([10, 255, 255])),
        (np.array([160, 50, 50]), np.array([180, 255, 255])),
    ],
    "green": [(np.array([40, 50, 50]), np.array([80, 255, 255]))],
    "blue": [(np.array([100, 50, 50]), np.array([130, 255, 255]))],
    "yellow": [(np.array([20, 50, 50]), np.array([35, 255, 255]))],
}


def decode_jpeg(jpeg_bytes):
    image = cv2.imdecode(np.frombuffer(jpeg_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("Could not decode JPEG bytes.")
    return image


def detect_color_blobs(jpeg_bytes, min_area=MIN_BLOB_AREA):
    image = decode_jpeg(jpeg_bytes)
    height, width = image.shape[:2]
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    detections = []

    for color, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for low, high in ranges:
            mask |= cv2.inRange(hsv, low, high)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < min_area:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            cx = x + w / 2
            angle_deg = ((cx / width) - 0.5) * 2 * HFOV_HALF_DEG
            detections.append({
                "color": color,
                "bbox": (x, y, w, h),
                "center": (cx, y + h / 2),
                "angle_deg": angle_deg,
                "blob_area": area,
            })
    return detections


async def perceive():
    jpeg = await get_camera_frame()
    return detect_color_blobs(jpeg)


async def scan_head(yaws=(-0.8, 0.0, 0.8), pitch=0.15):
    all_detections = []
    for yaw in yaws:
        await set_head(yaw=yaw, pitch=pitch)
        await asyncio.sleep(0.4)
        all_detections.extend(await perceive())
    return all_detections


## Student TODO: Agent Design

아래 셀들이 실제 프로젝트 작업 영역입니다: LLM prompting, observation, navigation/action execution, verification, recovery, memory를 설계하세요.

In [ ]:
# ---------------------------------------------------------------------------
# STUDENT TODO: LLM decision, observation, verification, memory
# ---------------------------------------------------------------------------

async def decide_next_action(task, observation, memory, last_result=None):
    """TODO: replace the fallback with a real text LLM decision call.

    Your LLM should return JSON with next_action, target_color, and reason.
    Validate the output with parse_agent_decision before executing actions.
    """
    decision_context = build_decision_context(task, observation, memory, last_result)

    # Prompt 예시:
    # system: Return ONLY JSON using this schema:
    # {"next_action": "search_cube", "target_color": "red", "reason": "..."}
    # user: json.dumps(decision_context)

    visible = decision_context["visible_targets"]
    if not visible:
        return AgentDecision(next_action="search_cube", reason="Fallback: no visible target.")
    largest = max(visible, key=lambda item: item["blob_area"])
    return AgentDecision(
        next_action="navigate_to_cube",
        target_color=largest["color"],
        reason="Fallback: choose the largest visible color blob.",
    )


async def observe_world(memory):
    """TODO: decide what the agent observes before asking the LLM.

    Add scan strategy, VLM summaries, confidence, or search notes if useful.
    Do not use scene_state or exact entity IDs in submitted code.
    """
    robot_status = await get_robot_status()
    detections = await scan_head()
    return Observation(robot_status=robot_status, detections=detections)


async def verify_outcome(decision, action_result):
    """TODO: re-observe and decide whether the last action worked."""
    return {"decision": decision.__dict__, "action_result": action_result}


def update_memory(memory, observation, decision, verified):
    """TODO: track progress, failures, held color, and useful presentation logs."""
    memory.logs.append({
        "observation": {"visible_count": len(observation.detections), "note": observation.note},
        "llm_decision": decision.__dict__,
        "verified": verified,
    })


In [ ]:
# ---------------------------------------------------------------------------
# LEVEL 2 TODO: vision-only action implementation
# ---------------------------------------------------------------------------
# Level 2에서는 go_to를 호출하면 안 됩니다. set_head, camera observation, set_velocity,
# memory, recovery behavior로 이동해야 합니다.


async def visual_search(target_color=None):
    """TODO: search for a cube or pad using camera movement and robot motion."""
    await scan_head()
    return False


async def visual_navigate_to_target(target_color):
    """TODO: implement closed-loop vision-only navigation.

    Observe -> move briefly -> observe again -> correct or stop.
    Handle target loss, overshoot, and obstacles.
    """
    return False


async def recover_motion(memory, reason=None):
    """TODO: recover from target loss, blocked motion, or failed manipulation."""
    await move_velocity(vx=-0.15, duration_s=0.8)
    await move_velocity(wz=0.35, duration_s=0.8)
    return {"action": "recover", "reason": reason, "status": "stepped_back_and_rotated"}


async def execute_decision(decision, observation, memory):
    """TODO: translate one validated LLM decision into a Level 2 action."""
    if decision.next_action in {"search_cube", "search_pad"}:
        found = await visual_search(decision.target_color)
        return {"action": decision.next_action, "found": found}

    if decision.next_action in {"navigate_to_cube", "navigate_to_pad"}:
        reached = await visual_navigate_to_target(decision.target_color)
        return {"action": decision.next_action, "reached": reached}

    if decision.next_action == "pick_cube":
        return {"action": "pick_cube", "result": result_summary(await pick_nearest_cube())}

    if decision.next_action == "place_cube":
        return {"action": "place_cube", "result": result_summary(await place_nearest_zone())}

    if decision.next_action == "recover":
        return await recover_motion(memory, decision.recovery_strategy)

    return {"action": decision.next_action, "status": "no_op"}


In [ ]:
async def run_agent(task, max_cycles=20):
    """얇은 observe-LLM-act loop입니다. loop만이 아니라 TODO 함수들을 수정하세요."""
    memory = AgentMemory()
    last_result = None

    for cycle in range(1, max_cycles + 1):
        print(f"\nCycle {cycle}")
        observation = await observe_world(memory)
        decision = await decide_next_action(task, observation, memory, last_result)
        print("LLM decision:", decision)

        if decision.next_action == "stop":
            break

        action_result = await execute_decision(decision, observation, memory)
        verified = await verify_outcome(decision, action_result)
        update_memory(memory, observation, decision, verified)
        last_result = verified

    return memory


## Run

현재 실험에 필요한 TODO를 충분히 구현한 뒤 starter loop를 실행하세요.

In [ ]:
memory = await run_agent(TASK)
memory.logs


In [ ]:
# Optional cleanup when finished.
# await session.close()
# await client.delete_robot(robot_id)
